# GLiNER fine-tune — Bosch GDPR PII (Colab)

Runs on a free Colab T4 in ~30-60min for 3 epochs over ~3.7K examples.

## Prep on your Mac (before opening this in Colab)

```bash
cd /path/to/west-monroe
git archive --format=zip HEAD -o /tmp/tracer.zip
zip -r /tmp/gdpr-data.zip data/out data/gliner
```

## In Colab

1. Runtime → Change runtime type → **T4 GPU**
2. Left sidebar → **Files icon** (folder) → drag-drop both zips into `/content/`
3. Run cells top-to-bottom

Trained model lands at `/content/drive/MyDrive/gliner-bosch/`.

In [ ]:
# 1. GPU + uploads check
import os
%cd /content
print('--- GPU ---')
!nvidia-smi 2>/dev/null | head -5 || print('NO GPU — switch runtime to T4 GPU')
print('--- uploads ---')
for z in ['/content/tracer.zip', '/content/gdpr-data.zip']:
    print(f"{z}: {'OK' if os.path.isfile(z) else 'MISSING — drag into Files panel'}")

In [ ]:
# 2. Extract zips → /content/repo (cwd-safe; refuses if uploads missing)
import os, sys, shutil

REPO_DIR = '/content/repo'
TRACER_ZIP = '/content/tracer.zip'
DATA_ZIP = '/content/gdpr-data.zip'

os.chdir('/content')

assert os.path.isfile(TRACER_ZIP), f'{TRACER_ZIP} missing — drag-drop into Files panel'
assert os.path.isfile(DATA_ZIP), f'{DATA_ZIP} missing — drag-drop into Files panel'

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.makedirs(REPO_DIR, exist_ok=True)

!unzip -q $TRACER_ZIP -d $REPO_DIR
!unzip -q $DATA_ZIP   -d $REPO_DIR

# Verify package layout
for p in ['train/__init__.py', 'train/finetune.py', 'data_gen/__init__.py', 'pyproject.toml']:
    full = os.path.join(REPO_DIR, p)
    assert os.path.isfile(full), f'extraction broken: {full} missing'
print('repo extracted OK')
print('--- contents ---')
!ls $REPO_DIR

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR
os.chdir(REPO_DIR)

In [ ]:
# 3. Install training deps (torch already on Colab)
import os
REPO_DIR = '/content/repo'
os.chdir(REPO_DIR)
!pip install -q 'gliner>=0.2.13' 'transformers>=4.40.0' 'accelerate>=0.30.0' 'seqeval>=1.2.2' orjson faker jinja2 pydantic httpx python-dotenv tqdm reportlab python-docx pypdf
print('--- imports check ---')
import gliner, transformers, accelerate
print(f'gliner={gliner.__version__}  transformers={transformers.__version__}  accelerate={accelerate.__version__}')

In [ ]:
# 4. Verify GLiNER token data present (uploaded zip should already include data/gliner/)
import os
REPO_DIR = '/content/repo'
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

needed = ['data/gliner/train.jsonl', 'data/gliner/val.jsonl']
missing = [p for p in needed if not os.path.isfile(os.path.join(REPO_DIR, p))]

if missing:
    print(f'Missing: {missing}\nRunning convert from data/out ...')
    assert os.path.isfile(f'{REPO_DIR}/data/out/train.jsonl'), 'data/out/train.jsonl also missing — re-zip data/out + re-upload'
    !PYTHONPATH=$REPO_DIR python -m train.convert --in-dir data/out --out-dir data/gliner
else:
    print('data/gliner/ ready')

!ls -lh $REPO_DIR/data/gliner/

In [ ]:
# 5. Mount Google Drive so the model survives session disconnect
import os
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/gliner-bosch'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'OUT_DIR={OUT_DIR}')

In [ ]:
# 6. Train
import os
REPO_DIR = '/content/repo'
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

# Pre-flight asserts (fail fast w/ clear msg)
assert os.path.isfile(f'{REPO_DIR}/train/finetune.py'), 'train/finetune.py missing — re-run cell 2'
assert os.path.isfile(f'{REPO_DIR}/data/gliner/train.jsonl'), 'data/gliner/train.jsonl missing — re-run cell 4'
assert 'OUT_DIR' in dir(), 'OUT_DIR undefined — run cell 5 (mount Drive) first'

print(f'cwd={os.getcwd()}  PYTHONPATH={os.environ["PYTHONPATH"]}  OUT_DIR={OUT_DIR}')

!PYTHONPATH=$REPO_DIR python -m train.finetune \
    --train data/gliner/train.jsonl \
    --val data/gliner/val.jsonl \
    --base-model urchade/gliner_multi_pii-v1 \
    --out $OUT_DIR \
    --epochs 3 \
    --batch-size 16 \
    --lr 5e-6 \
    --lr-others 1e-5

In [ ]:
# 7. Sanity-check the trained model on a held-out example
from gliner import GLiNER
model = GLiNER.from_pretrained(OUT_DIR, local_files_only=True)
labels = ['PERSON', 'EMPLOYEE_ID', 'EMAIL', 'PHONE', 'ADDRESS', 'TAX_ID', 'IBAN', 'DEPARTMENT', 'COMPANY', 'DATE', 'SIGNATURE', 'USERNAME']
sample = (
    'Spesenabrechnung (ausgefüllt)\n'
    'Mitarbeiter: Hans Müller (E-43217)\n'
    'Abteilung: Forschung & Entwicklung\n'
    'Datum: 12 Mär 2026\n'
    'Vorgesetzter: Anna Becker\n'
    'Unterschrift: A. Becker'
)
for ent in model.predict_entities(sample, labels, threshold=0.4):
    print(f"{ent['label']:<14} {ent['text']!r}  (score={ent['score']:.2f})")

In [ ]:
# 8. Zip + download checkpoint for use in scan service
import shutil
shutil.make_archive('/content/gliner-bosch', 'zip', OUT_DIR)
from google.colab import files
files.download('/content/gliner-bosch.zip')